# 3 · `EngramStore` — a LangGraph `BaseStore`

`EngramStore` exposes Engram through LangGraph's `BaseStore` API
(`put` / `get` / `search` / `delete`), so existing store-based code — including
`create_agent(store=...)` and LangMem — can target Engram.

> **Experimental.** Engram is a memory-*extraction* service, not an exact
> key-value store. `put` is processed asynchronously and Engram assigns its own
> deduped ids, so the key you write is **not** the key you read back. Use
> `search` to retrieve, then `get`/`delete` by the returned key.
> Namespaces map as `(user_id,)` or `(user_id, group)`.

## Setup — pick **one** option

**Option A — editable install (recommended).** Installs this package *and* its
dependencies from source; edits to `langchain_engram/` are picked up after a
kernel restart.

**Option B — import from local source (no install of `langchain-engram`).** Adds
the repo root to `sys.path` so `import langchain_engram` resolves straight to the
source tree. You still need the runtime dependencies available — the easiest is to
launch Jupyter from the repo's environment: `uv run --with jupyter jupyter lab`.

In [ ]:
# Option A — editable install from source.
%pip install -q -e ".."

In [ ]:
# Option B — use langchain-engram straight from the local source tree (no install).
# Run this INSTEAD of Option A.
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebooks/ -> repo root (holds langchain_engram/)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import langchain_engram

print('langchain_engram imported from:', langchain_engram.__file__)

In [ ]:
import getpass
import os

if not os.environ.get("ENGRAM_API_KEY"):
    os.environ["ENGRAM_API_KEY"] = getpass.getpass("ENGRAM_API_KEY: ")

# Optional: point at a non-default Engram endpoint.
# os.environ["ENGRAM_BASE_URL"] = "https://..."

In [ ]:
import time

from langchain_engram._client import build_client

client = build_client()  # reads ENGRAM_API_KEY


def seed_memory(text, user_id, **kwargs):
    """Add a memory and block until the async pipeline commits it."""
    run = client.memories.add(text, user_id=user_id, **kwargs)
    status = client.runs.wait(run.run_id, timeout=60.0)
    print(f'run {run.run_id}: {status.status} '
          f'(+{len(status.memories_created)} / ~{len(status.memories_updated)})')
    return status


def wait_until_searchable(query, user_id, needle, timeout=60.0, **kwargs):
    """Poll search until `needle` shows up (handles eventual consistency)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        results = client.memories.search(query=query, user_id=user_id, **kwargs)
        if any(needle.lower() in m.content.lower() for m in results):
            return results
        time.sleep(2.0)
    return client.memories.search(query=query, user_id=user_id, **kwargs)

### Put then search
Namespace `(user_id,)` scopes the memory to a user.

In [ ]:
from langchain_engram import EngramStore

store = EngramStore()
USER = 'store-user'

store.put((USER,), key='ignored-by-engram',
          value={'content': 'The user manages a rooftop garden.'})

# eventual consistency: poll until the pipeline commits
found = wait_until_searchable('gardening', USER, 'garden')
items = store.search((USER,), query='gardening', limit=5)
for it in items:
    print(it.key, '->', it.value['content'], f'(score={it.score})')

### Get and delete by the returned key
The `key` on a search hit is the Engram memory id.

In [ ]:
if items:
    memory_id = items[0].key
    fetched = store.get((USER,), memory_id)
    print('get:', fetched.value['content'] if fetched else None)

    store.delete((USER,), memory_id)
    print('deleted', memory_id)

### Use it as an agent's long-term store
Any agent code that expects a LangGraph store now writes to Engram.

In [ ]:
from langchain.agents import create_agent

agent = create_agent('anthropic:claude-sonnet-4-6', store=EngramStore())
# (requires ANTHROPIC_API_KEY; see notebook 1)
print(type(agent))

### Namespace rules
`(user_id,)` and `(user_id, group)` are valid; deeper namespaces raise.

In [ ]:
try:
    store.put(('u', 'g', 'extra'), key='k', value={'content': 'x'})
except ValueError as e:
    print('rejected:', e)